# Tipos `Annotated` e variáveis de tipo

Tecnicamente isto não é específico do Pydantic, mas é uma técnica muito usada para definir tipos em modelos Pydantic.

Suponha que queremos uma lista limitada a, no máximo, 10 elementos.

Já vimos como fazer isso com a classe `Field`:

In [1]:
from typing import Annotated

from pydantic import BaseModel, Field, ValidationError

In [2]:
class Model(BaseModel):
    elements: list[int] = Field(default=[], max_length=10)

In [3]:
m = Model(elements=[1, 2, 3])
print(f"m: {m!r}")

m: Model(elements=[1, 2, 3])


In [4]:
try:
    Model(elements = [1, ] * 20)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
elements
  List should have at most 10 items after validation, not 20 [type=too_long, input_value=[1, 1, 1, 1, 1, 1, 1, 1, ... 1, 1, 1, 1, 1, 1, 1, 1], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


Para reutilizar, poderíamos definir isso como tipo anotado:

In [5]:
BoundedListInt = Annotated[list[int], Field(max_length=10)]

E reutilizá-lo em qualquer modelo:

In [6]:
class Model(BaseModel):
    field_1: BoundedListInt = []
    field_2: BoundedListInt = []

Ótimo, mas esse tipo anotado só serve para listas de inteiros.

E se quisermos o mesmo para listas de floats?

In [7]:
BoundedListFloat = Annotated[list[float], Field(max_length=10)]

E para strings?

In [8]:
BoundedListString = Annotated[list[str], Field(max_length=10)]

E assim por diante...

A única coisa que muda é o tipo dentro de `list[]`: `int`, `float`, `str`. Para outros tipos, teríamos de continuar criando tipos que, fora
isso, são idênticos.

Poderíamos usar `Any` na anotação — mas perderíamos a validação de tipo do Pydantic (a lista aceitaria qualquer coisa, inclusive tipos
misturados), o que provavelmente não queremos.

In [9]:
from typing import Any

BoundedList = Annotated[list[Any], Field(max_length=10)]

É aqui que entram as **variáveis de tipo**.

Pense em como definimos uma lista de inteiros ou de strings...

Usamos o tipo `list`, colchetes e o tipo dos elementos dentro dos colchetes, ex.: `list[int]`.

Esse tipo dos elementos poderia ser qualquer um, até uma classe nossa.

Ou seja, o tipo dentro dos colchetes é, na verdade, **variável**.

Esse é o conceito de variável de tipo, implementado em Python com `TypeVar`.

Vejamos como fazer isso com nossos próprios tipos.

Primeiro definimos uma variável de tipo — o nome é livre; aqui usamos `T`.

In [10]:
from typing import TypeVar

In [11]:
T = TypeVar('T')

Agora usamos essa variável no tipo anotado, no lugar de `int` ou `float`.

In [12]:
BoundedList = Annotated[list[T], Field(max_length=10)]

Para criar uma lista limitada de um tipo específico, basta informar o valor da variável de tipo ao usar o tipo anotado:

In [13]:
print(f"BoundedList[int]: {BoundedList[int]!r}")

BoundedList[int]: typing.Annotated[list[int], FieldInfo(annotation=NoneType, required=True, metadata=[MaxLen(max_length=10)])]


In [14]:
print(f"BoundedList[str]: {BoundedList[str]!r}")

BoundedList[str]: typing.Annotated[list[str], FieldInfo(annotation=NoneType, required=True, metadata=[MaxLen(max_length=10)])]


E usamos nos modelos:

In [15]:
class Model(BaseModel):
    integers: BoundedList[int] = []
    strings: BoundedList[str] = []

In [16]:
print(f"Model(): {Model()!r}")

Model(): Model(integers=[], strings=[])


In [17]:
print(f'Model(integers=[1.0, 2.0], strings=["abc", "def"]): {Model(integers=[1.0, 2.0], strings=["abc", "def"])!r}')

Model(integers=[1.0, 2.0], strings=["abc", "def"]): Model(integers=[1, 2], strings=['abc', 'def'])


E os elementos da lista têm tipo específico:

In [18]:
try:
    Model(integers=[0.5])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
integers.0
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/int_from_float


Dá para fazer muito mais com variáveis de tipo, mas este não é um curso de *typing* — a técnica foi mostrada porque é muito útil para
definir tipos Pydantic reutilizáveis.

> 💡 **Complemento (Python 3.12+):** a sintaxe nova de alias de tipo (PEP 695) dispensa o `TypeVar` explícito, e o Pydantic a entende:

In [19]:
type BoundedList2[T] = Annotated[list[T], Field(max_length=3)]

class Model2(BaseModel):
    integers: BoundedList2[int] = []

try:
    Model2(integers=[1, 2, 3, 4])
except ValidationError as ex:
    print(f'ex.errors()[0]["type"]: {ex.errors()[0]["type"]}')

ex.errors()[0]["type"]: too_long
